In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import RFECV
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
%matplotlib inline


In [2]:
train = pd.read_pickle('../datos/procesados/train_transformado.pkl')

# Preselcción de variables

### Mutual selector

In [3]:
x = train.drop(columns=['estado_prestamo'])
y = train['estado_prestamo']

In [4]:
def ranking_mi(mutual_selector, modo = 'tabla'):
    #Maqueta el ranking
    ranking_mi = pd.DataFrame(mutual_selector, index = x.columns).reset_index()
    ranking_mi.columns = ['variable','importancia_mi']
    ranking_mi = ranking_mi.sort_values(by = 'importancia_mi', ascending = False)
    ranking_mi['ranking_mi'] = np.arange(0,ranking_mi.shape[0])
    #Muestra la salida
    if modo == 'tabla':
        return(ranking_mi)
    else:
        g = ranking_mi[0:15].importancia_mi.sort_values().plot.barh()
        g.set_yticklabels(ranking_mi[0:15].sort_values(by = 'importancia_mi').variable)
        return(g)

In [5]:
mutual_selector = mutual_info_classif(x,y)
ranking_mi(mutual_selector)

,variable,importancia_mi,ranking_mi
1,ingreso,0.109545,0
6,porcentaje_ingreso,0.083007,1
5,tasa_interes,0.079195,2
3,calificacion_prestamo,0.079139,3
11,propiedad_vivienda_RENT,0.030092,4
9,propiedad_vivienda_MORTGAGE,0.019905,5
4,monto_prestamo,0.017006,6
7,incumplimiento_historial,0.016663,7
10,propiedad_vivienda_OWN,0.004212,8
12,intencion_prestamo_DEBTCONSOLIDATION,0.003910,9


Eliminar las que den 0 ya que no tienen relaciones con la target

In [6]:
train.drop(columns=['duracion_credito','intencion_prestamo_HOMEIMPROVEMENT'], inplace=True)

### Correlaciones

In [7]:
def correlaciones_fuertes(df, lim_inf = 0.3, lim_sup = 1,drop_dupli=True):
    #Calcula la matriz de correlación
    c = df.corr().abs()
    #Lo pasa todo a filas
    c= c.unstack()
    #Pasa el índice a columnas y le pone nombres
    c = pd.DataFrame(c).reset_index()
    c.columns = ['var1','var2','corr']
    #A dataframe, filtra limites y ordena en descendiente
    c = c.loc[(c['corr'] > lim_inf) &  (c['corr'] < lim_sup),:].sort_values(by = 'corr', ascending=False)
    #Desduplica las correlaciones (o no si drop_dupli es False)
    c = c if drop_dupli == False else c.drop_duplicates(subset = ['corr'])
    #Devuelve la salida
    return(c)

In [8]:
correlaciones_fuertes(train)

,var1,var2,corr
56,calificacion_prestamo,tasa_interes,0.933330
196,propiedad_vivienda_RENT,propiedad_vivienda_MORTGAGE,0.855884
123,porcentaje_ingreso,monto_prestamo,0.647602
59,calificacion_prestamo,incumplimiento_historial,0.536140
93,tasa_interes,incumplimiento_historial,0.500142
21,ingreso,monto_prestamo,0.383795
57,calificacion_prestamo,estado_prestamo,0.379977
125,porcentaje_ingreso,estado_prestamo,0.379368
91,tasa_interes,estado_prestamo,0.340751
120,porcentaje_ingreso,ingreso,0.325585


Eliminar las correlaciones en base a la información mutua y a las correlaciones fuertes.

In [9]:
train.drop(columns=['calificacion_prestamo','propiedad_vivienda_MORTGAGE'], inplace=True)

### Recursive Feature Elimination

In [10]:
rfe = RFECV(estimator = XGBClassifier(n_jobs = -1, eval_metric='auc'),
            scoring = 'roc_auc',
            n_jobs = -1)

rfe.fit(x,y)

rank_rfe = pd.DataFrame({'variable': x.columns, 'ranking_rfe': rfe.ranking_}).sort_values(by = 'ranking_rfe')
rank_rfe

,variable,ranking_rfe
1,ingreso,1
2,duracion_empleo,1
3,calificacion_prestamo,1
6,porcentaje_ingreso,1
12,intencion_prestamo_DEBTCONSOLIDATION,1
11,propiedad_vivienda_RENT,1
10,propiedad_vivienda_OWN,1
9,propiedad_vivienda_MORTGAGE,1
15,intencion_prestamo_MEDICAL,1
14,intencion_prestamo_HOMEIMPROVEMENT,1


Calificar las variales con RFE y quitar las no principales (distintas a 1)

In [11]:
train.drop(columns=['duracion_empleo','incumplimiento_historial','edad','monto_prestamo'], inplace=True)

### Permutation importance

In [12]:
x = train.drop(columns=['estado_prestamo'])
y = train['estado_prestamo']

xgb = XGBClassifier(n_jobs = -1, eval_metric='auc')

xgb.fit(x,y)

permutacion = permutation_importance(xgb, 
                                     x, y, 
                                     scoring = 'roc_auc',
                                     n_repeats=5, n_jobs = -1)

In [13]:
def ranking_per(predictoras,permutacion):
    ranking_per = pd.DataFrame({'variable': predictoras.columns, 'importancia_per': permutacion.importances_mean}).sort_values(by = 'importancia_per', ascending = False)
    ranking_per['ranking_per'] = np.arange(0,ranking_per.shape[0])
    return(ranking_per)

In [14]:
ranking_per(x, permutacion)

,variable,importancia_per,ranking_per
2,porcentaje_ingreso,0.144937,0
1,tasa_interes,0.137007,1
0,ingreso,0.118757,2
4,propiedad_vivienda_RENT,0.051157,3
3,propiedad_vivienda_OWN,0.026532,4
9,intencion_prestamo_VENTURE,0.024448,5
7,intencion_prestamo_MEDICAL,0.013075,6
5,intencion_prestamo_DEBTCONSOLIDATION,0.009887,7
6,intencion_prestamo_EDUCATION,0.008200,8
8,intencion_prestamo_PERSONAL,0.007293,9


# Balanceo de clases

In [15]:
train.estado_prestamo.value_counts()

estado_prestamo
0    21060
1     6029
Name: count, dtype: int64

No aplicamos balanceo de clases

# Exportar el dataset 

In [16]:
X = train.drop(columns=['estado_prestamo'])
y = train['estado_prestamo']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
train.to_pickle('../datos/procesados/train_preseleccionado.pkl')
X_train.to_pickle('../datos/entrenamiento/X_train.pkl')
X_test.to_pickle('../datos/entrenamiento/X_test.pkl')
y_train.to_pickle('../datos/entrenamiento/y_train.pkl')
y_test.to_pickle('../datos/entrenamiento/y_test.pkl')